# TotalVI RNA + protein — Patient 1 (03H096 / PB2)

**Role in the paper:** Denoised ADT abundances used to design the FACS gates
and the RNA/protein differential-expression tables.

**What this notebook does**
1. Loads the raw MuData and restricts it to the barcodes kept by MultiVI
2. Runs RNA, ATAC and protein quality control
3. Keeps the transcription-factor transcripts (Ensembl TF list) and the
   informative antibodies
4. Trains TotalVI on RNA + protein and extracts denoised expression
5. Transfers the MultiVI clusters and ForceAtlas layout onto the TotalVI object
6. Runs differential expression per cluster and exports the tables
7. Writes the TotalVI MuData used by the figure notebooks

**Objects**
- **Reads:** `DATA_DIR / "Teaseq_PB2.h5mu"`,
  `DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad"`,
  `DATA_DIR / "resources/TFs_Ensembl_v_1.01.txt"`
- **Creates:** `DATA_DIR / "03_RNA_and_protein_TotalVI/objects/PB2/TotalVI_PB2_TEAseq_CLEAN_TF.h5mu"`,
  the trained model directory and the DE tables under
  `03_RNA_and_protein_TotalVI/PB2/DE/`

The highly-variable-gene variant of the same object
(`TotalVI_PB2_TEAseq_CLEAN_HV.h5mu`) is produced by re-running this notebook
without the transcription-factor filter.

## Paths and settings

In [ ]:
from pathlib import Path

# Root of the companion data package. Point this at your local copy.
DATA_DIR = Path("PATH_TO_DATA")  # <-- set this to your local data root
OUT_DIR = DATA_DIR / "outputs/TotalVI_PB2"     # figures and tables written by this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import os

import anndata as ad
import matplotlib.pyplot as plt
import mudata as md
import muon
import numpy as np
import pandas as pd
import scanpy as sc
import scvi

## Load the raw MuData and the cleaned MultiVI reference

In [ ]:
### Read the H5 file ###

adataPB2 = muon.read(DATA_DIR / "Teaseq_PB2.h5mu")
adataPB2.var_names_make_unique()

In [ ]:
# Cleaned MultiVI object: source of the barcode list and cluster labels.
adataMultiPB2 = ad.read_h5ad(
    DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad"
)

## Harmonise antibody and cell names

In [ ]:
# Normalise the antibody names to the ADT.anti.hu.<clone> convention.
adataPB2.mod['protein'].var.index = 'ADT.anti.hu.' + adataPB2.mod['protein'].var.index.str.split('-').str[0]

In [ ]:
### Annotation of all the samples ###

adataPB2.obs.index = [name + '_PB2' for name in adataPB2.obs_names]

In [ ]:
adata = adataPB2

## Split modalities and restrict to the QC-passing barcodes

In [ ]:
### Extracting RNA data only ###

rna_adata = adata.mod['rna']
prot_adata = adata.mod['protein']
atac_adata = adata.mod['atac']
rna_adata.obs.index = adata.obs.index
prot_adata.obs.index = adata.obs.index
atac_adata.obs.index = adata.obs.index

In [ ]:
# Barcodes retained by MultiVI; every modality is subset to this list.
cell_ids_list = adataMultiPB2.obs_names.tolist()
print(len(cell_ids_list), 'cells;', cell_ids_list[:3])

In [ ]:
# Filter the MuData object
rna_adata = rna_adata[rna_adata.obs.index.isin(cell_ids_list)]
rna_adata

In [ ]:
# Filter the MuData object
prot_adata = prot_adata[prot_adata.obs.index.isin(cell_ids_list)]
prot_adata

In [ ]:
# Filter the MuData object
atac_adata = atac_adata[atac_adata.obs.index.isin(cell_ids_list)]
atac_adata

## ATAC quality control

In [ ]:
### Three plot summary BEFORE filtering ###
import scanpy as sc

sc.pp.calculate_qc_metrics(atac_adata, percent_top=None, log1p=False, inplace=True)

atac_adata.var_names_make_unique()

sc.pl.violin(atac_adata, ['n_genes_by_counts', 'total_counts'],
             jitter=0.4, multi_panel=True)

atac_adata

In [ ]:
atac_adata.obs['cluster']=adataMultiPB2.obs['Cluster']

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cluster_key = 'cluster'  # MultiVI clusters copied onto the ATAC modality

# Compute the mean n_genes_by_counts per cluster
mean_n_genes = atac_adata.obs.groupby(cluster_key)['n_genes_by_counts'].mean().sort_values()

# Plot the bar plot
plt.figure(figsize=(10, 6))
mean_n_genes.plot(kind='bar', color='skyblue', edgecolor='black')
plt.ylabel('Mean accessible ATAC peaks per cell')
plt.xlabel('Cluster')
plt.title('Accessible ATAC peaks per cell by cluster')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## RNA quality control

In [ ]:
rna_adata.obs['Cluster_Final']=adataMultiPB2.obs['Cluster_Final']

In [ ]:
### Top ranking expressed gene ###

sc.pl.highest_expr_genes(rna_adata, n_top=20, )

In [ ]:
### Three plot summary BEFORE filtering ###

rna_adata.var['mt'] = rna_adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(rna_adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

rna_adata.var_names_make_unique()

sc.pl.violin(rna_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

rna_adata

In [ ]:
import os

# Define output directory
outdir = DATA_DIR / "metrics/PB2"

# Create directory if it does not exist
os.makedirs(outdir, exist_ok=True)

# Define file path
outfile = os.path.join(outdir, "rna_adata.h5ad")

# Save
rna_adata.write(outfile)

print("Saved to:", outfile)

In [ ]:
### Summary of genes and counts BEFORE filtering ###

sc.pl.scatter(rna_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(rna_adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
# RNA cell/feature filters as reported in Methods. Most cells already
# pass them because the barcode list was restricted above.

sc.pp.filter_cells(rna_adata, min_genes=200)
sc.pp.filter_genes(rna_adata, min_cells=3)
rna_adata = rna_adata[rna_adata.obs.n_genes_by_counts < 5000, :]
rna_adata = rna_adata[rna_adata.obs.n_genes_by_counts > 350, :]
rna_adata = rna_adata[rna_adata.obs.pct_counts_mt < 40, :]

In [ ]:
### Three plot summary AFTER filtering ###

sc.pl.violin(rna_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

rna_adata

In [ ]:
### Summary of genes and counts AFTER filtering ###

sc.pl.scatter(rna_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(rna_adata, x='total_counts', y='n_genes_by_counts')

## Protein (ADT) quality control

In [ ]:
### Top ranking expressed gene ###

sc.pl.highest_expr_genes(prot_adata, n_top=20, )

In [ ]:
### Three plot summary BEFORE filtering ###

prot_adata.var['mt'] = prot_adata.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(prot_adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

prot_adata.var_names_make_unique()

sc.pl.violin(prot_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

prot_adata

In [ ]:
### Summary of genes and counts BEFORE filtering ###

sc.pl.scatter(prot_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(prot_adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
### Three plot summary AFTER filtering ###

sc.pl.violin(prot_adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

prot_adata

In [ ]:
prot_adata.obs['Cluster_Final']=rna_adata.obs['Cluster_Final']

sc.pl.violin(
    prot_adata,
    ['n_genes_by_counts', 'total_counts'],
    groupby='Cluster_Final',
    jitter=0.4,
    multi_panel=True,
    rotation=90
)

In [ ]:
### Summary of genes and counts AFTER filtering ###

sc.pl.scatter(prot_adata, x='total_counts', y='pct_counts_mt')
sc.pl.scatter(prot_adata, x='total_counts', y='n_genes_by_counts')

In [ ]:
# Filter out variables that contain 'IgG' in their names
prot_adata = prot_adata[:, ~prot_adata.var.index.str.contains('Rat|Mouse|Hamster')]
prot_adata

## Keep transcription-factor transcripts only

The TotalVI object is restricted to annotated human transcription factors so that the DE tables focus on regulators. Skip this cell to obtain the highly-variable-gene variant.

In [ ]:
import pandas as pd

# Load the Ensembl IDs from the text file
ensembl_ids_path = DATA_DIR / "resources/TFs_Ensembl_v_1.01.txt"
ensembl_ids_df = pd.read_csv(ensembl_ids_path, header=None, names=['Ensembl_ID'])
ensembl_ids = ensembl_ids_df['Ensembl_ID'].tolist()

In [ ]:
# Filter the AnnData object
rna_adata = rna_adata[:, rna_adata.var['gene_ids'].isin(ensembl_ids)]

rna_adata

## Assemble the RNA + protein MuData

In [ ]:
# Find the common observations
common_obs = rna_adata.obs_names.intersection(prot_adata.obs_names)

# Subset the rna_adata and prot_adata objects to keep only the common observations
rna_adata = rna_adata[common_obs]
prot_adata = prot_adata[common_obs]

In [ ]:
### Add the protein information in the Anndata object ###

adata=rna_adata

# Get the cell names of the filtered RNA data
cell_names = rna_adata.obs_names

# Add the protein expression data to the obsm attribute of the RNA AnnData object
rna_adata.obsm['protein_expression'] = prot_adata.X

# Add the names of the protein markers to the uns attribute of the RNA AnnData object
rna_adata.uns['protein_names'] = prot_adata.var_names

adata=rna_adata

adata

In [ ]:
### Normalization of the data ###

adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.obs_names_make_unique()

In [ ]:
### Remergeed the protein data with the rna data ###

# Create a new AnnData object for the protein expression data
protein_adata = ad.AnnData(adata.obsm["protein_expression"])

# Set the obs_names of the protein AnnData object to match the RNA AnnData object
protein_adata.obs_names = adata.obs_names

# Set the var_names of the protein AnnData object to the names of the protein markers
protein_adata.var_names = adata.uns['protein_names']

# Remove the protein expression data from the obsm attribute of the RNA AnnData object
del adata.obsm["protein_expression"]

# Create a MuData object containing both the RNA and protein data
mdata = md.MuData({"rna": adata, "protein": protein_adata})

# Extract the batch information from the cell identifiers
batches = [cell_id.split('_')[-1] for cell_id in mdata.obs_names]

# Add the batch information to the rna modality
mdata.mod['rna'].obs['batch'] = pd.Categorical(batches)

mdata

In [ ]:
### Select the high variable gene ###

sc.pp.highly_variable_genes(
    mdata.mod["rna"],
    n_top_genes=4000,
    flavor="seurat_v3",
    batch_key="batch",
    layer="counts",
)
# Place subsetted counts in a new modality
mdata.mod["rna_subset"] = mdata.mod["rna"][
    :, mdata.mod["rna"].var["highly_variable"]
].copy()
mdata.update()
mdata

In [ ]:
### Check for duplicates ###

for modality in mdata.mod:
    print(f"Checking modality: {modality}")
    df = mdata.mod[modality].to_df()
    if len(df.columns) != len(df.columns.unique()):
        print(f"There are duplicate column names in modality {modality}")
        duplicates = df.columns[df.columns.duplicated()].unique()
        print(f"The following columns are duplicated: {duplicates}")
    else:
        print(f"There are no duplicate column names in modality {modality}")

## Train TotalVI

In [ ]:
### Set-up the total VI model ###

# Convert the protein expression data to a dense matrix
mdata.mod['protein'].X = mdata.mod['protein'].X.toarray()

mdata.update()

scvi.model.TOTALVI.setup_mudata(
    mdata,
    rna_layer="counts",
    protein_layer=None,
    batch_key=None,
    modalities={
        "rna_layer": "rna_subset",
        "protein_layer": "protein",
        "batch_key": "rna_subset",
    },
)

# Create the totalVI model
vae = scvi.model.TOTALVI(mdata)

In [ ]:
### Train the model ###

vae.train(max_epochs=400)

## Transfer the MultiVI clusters and ForceAtlas layout

In [ ]:
adata = ad.read_h5ad(DATA_DIR / "01_Trimodal_integration_MultiVI/cleaned_MultiVI/Teaseq_Multi_VI_PB2_Cleaned.h5ad")
adata

In [ ]:
muon.pl.embedding(
    adata,
    basis="X_draw_graph_fa",
    color=["Cluster_Final"],
    frameon=False,
    ncols=1,
)

In [ ]:
sc.pl.draw_graph(adata, 
                 size=200,
                 color = 'Cluster_Final')

In [ ]:
mdata.mod['rna_subset'].obs['leiden_totalVI']=adata.obs['Cluster_Final']
mdata.mod['rna_subset'].obsm['X_draw_graph_fa']=adata.obsm['X_draw_graph_fa']
mdata.update()

In [ ]:
### Plot the FA leiden_totalVI clusters ###

muon.pl.embedding(
    mdata,
    basis="rna_subset:draw_graph_fa",
    color=["rna_subset:n_genes_by_counts"],
    frameon=False,
    ncols=1,
)

## Denoised expression and latent space

In [ ]:
rna = mdata.mod["rna_subset"]
protein = mdata.mod["protein"]

rna.obsm["X_totalVI"] = vae.get_latent_representation()

rna_denoised, protein_denoised = vae.get_normalized_expression(
    n_samples=25, return_mean=True)

(
    rna.layers["denoised_rna"],
    protein.layers["denoised_protein"],
) = (rna_denoised, protein_denoised)

protein.layers["protein_foreground_prob"] = vae.get_protein_foreground_probability(
    n_samples=25, return_mean=True)
parsed_protein_names = [p.split("_")[0] for p in protein.var_names]
protein.var["clean_names"] = parsed_protein_names

mdata.update()

In [ ]:
### Plot the FA leiden_totalVI clusters ###

muon.pl.embedding(
    mdata,
    basis="rna_subset:draw_graph_fa",
    color=["rna_subset:leiden_totalVI"],
    frameon=False,
    ncols=1,
)

## Differential expression per cluster

In [ ]:
### Differential Expression Analysis ###

de_df = vae.differential_expression(
    groupby="rna_subset:leiden_totalVI", delta=0.5)
de_df.head(5)

In [ ]:
### Differential Expression SCVI Parameters ###

filtered_pro = {}
filtered_rna = {}

cats = rna.obs.leiden_totalVI.cat.categories

for i, c in enumerate(cats):
    cid = f"{c} vs Rest"
    cell_type_df = de_df.loc[de_df.comparison == cid]
    cell_type_df = cell_type_df.sort_values("lfc_median", ascending=False)

    # LFC filter
    cell_type_df = cell_type_df[cell_type_df.lfc_median > 0.2]

    pro_rows = cell_type_df.index.str.contains("hu")
    data_pro = cell_type_df.iloc[pro_rows]
    data_pro = data_pro[data_pro["bayes_factor"] > 0.7]

    data_rna = cell_type_df.iloc[~pro_rows]
    data_rna = data_rna[data_rna["bayes_factor"] > 1.5]
    data_rna = data_rna[data_rna["non_zeros_proportion1"] > 0.04]

    # Keep ALL filtered features (no top 1000 cutoff)
    filtered_pro[c] = data_pro.index.tolist()
    filtered_rna[c] = data_rna.index.tolist()

In [ ]:
### Dendrogram Setup ###

sc.tl.dendrogram(rna, groupby="leiden_totalVI", use_rep="X_totalVI")
# Resuse the RNA cluster labels and totalVI representation for the protein dendrogram
protein.obs["leiden_totalVI"] = rna.obs["leiden_totalVI"]
protein.obsm["X_totalVI"] = rna.obsm["X_totalVI"]
sc.tl.dendrogram(protein, groupby="leiden_totalVI", use_rep="X_totalVI")

In [ ]:
### Plot the heatmap with all proteins over totalVI clusters ###

sc.pl.matrixplot(
    protein,
    protein.var["clean_names"],
    groupby="leiden_totalVI",
    gene_symbols="clean_names",
    dendrogram=True,
    swap_axes=True,
    layer="denoised_protein",
    cmap="Greens",
    standard_scale="var",
)

In [ ]:
# Assuming filtered_rna_subset is already defined for your cluster of interest
sc.pl.dotplot(
    rna,
    filtered_rna,
    groupby="leiden_totalVI",
    dendrogram=True,
    standard_scale="var",
    swap_axes=True,
    dot_max=0.01,  # Adjust this value to increase or decrease the maximum dot size
    dot_min=0,  # Adjust this value to increase or decrease the minimum dot size
    color_map="Blues"  # Using a blue color map for colorblind accessibility
)

# Display the plot
plt.show()

In [ ]:
# Assuming filtered_rna_subset is already defined for your cluster of interest
sc.pl.dotplot(
    protein,
    filtered_pro,
    groupby="leiden_totalVI",
    dendrogram=True,
    standard_scale="var",
    swap_axes=True,
    dot_max=0.01,  # Adjust this value to increase or decrease the maximum dot size
    dot_min=0,  # Adjust this value to increase or decrease the minimum dot size
    color_map="Reds"  # Using a blue color map for colorblind accessibility
)

# Display the plot
plt.show()

## Save the model, the DE tables and the TotalVI object

Optional: if the model has already been trained, load it with `scvi.model.TOTALVI.load(...)` instead of re-running the training cell above.

In [ ]:
# Save the totalVI model and overwrite the existing directory
vae.save(DATA_DIR / "03_RNA_and_protein_TotalVI/PB2/Model", overwrite=True)

In [ ]:
# Save the data
de_df.to_csv(DATA_DIR / "03_RNA_and_protein_TotalVI/PB2/DE/de_df.csv")

import pandas as pd

# Convert dictionary to DataFrame
filtered_rna_df = pd.DataFrame(dict([ (k,pd.Series(v)) for k,v in filtered_rna.items() ]))

# Save DataFrame to csv
filtered_rna_df.to_csv(DATA_DIR / "03_RNA_and_protein_TotalVI/PB2/DE/filtered_rna.csv")

# Convert dictionary to DataFrame
filtered_pro_df = pd.DataFrame(dict([ (k,pd.Series(v)) for k,v in filtered_pro.items() ]))

# Save DataFrame to csv
filtered_pro_df.to_csv(DATA_DIR / "03_RNA_and_protein_TotalVI/PB2/DE/filtered_pro.csv")

In [ ]:
import pandas as pd

# Load your CSV file
file_path = DATA_DIR / "03_RNA_and_protein_TotalVI/PB2/DE/de_df.csv"
de_df = pd.read_csv(file_path)

# Create an Excel writer object
output_path = DATA_DIR / "03_RNA_and_protein_TotalVI/PB2/DE/de_df_by_cluster.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # Iterate through each cluster
    for cluster in de_df['comparison'].unique():
        # Filter the data for the current cluster
        cluster_data = de_df[de_df['comparison'] == cluster]
        # Write the cluster's data to a separate sheet
        cluster_data.to_excel(writer, sheet_name=cluster.replace(' ', '_'), index=False)

print(f"Data has been saved to {output_path}")

In [ ]:
import pandas as pd

# Load the DE DataFrame
file_path = DATA_DIR / "03_RNA_and_protein_TotalVI/PB2/DE/de_df.csv"
de_df = pd.read_csv(file_path)

# Filter and sort the data for '2 vs Rest'
cluster_of_interest = "2 vs Rest"
cluster_table = de_df[de_df["comparison"] == cluster_of_interest]

# Apply filtering logic (matching the one used for filtered_rna)
cluster_table = cluster_table[~cluster_table['Unnamed: 0'].str.contains('ADT')]  # Exclude proteins
cluster_table_filtered = cluster_table[
    (cluster_table["lfc_median"] > 0.2) &
    (cluster_table["bayes_factor"] > 1.5) &
    (cluster_table["non_zeros_proportion1"] > 0.04)
]
cluster_table_sorted = cluster_table_filtered.sort_values("lfc_median", ascending=False)

# Extract the top 50 gene names
top_50_gene_names_table = cluster_table_sorted['Unnamed: 0'].head(50).tolist()

# Print the top 50 gene names from the saved table
print("Top 50 gene names from the saved table (2 vs Rest) after applying filters:")
print(top_50_gene_names_table)

# Compare to the filtered_rna result
filtered_rna_top_50 = filtered_rna["2"][:50]  # Assuming this is already populated
print("\nTop 50 genes used in the dot plot:")
print(filtered_rna_top_50)

# Check if they match
matching = top_50_gene_names_table == filtered_rna_top_50
print("\nDo the two lists match?", matching)

In [ ]:
del mdata.mod['rna'].uns['protein_names']
del mdata.mod['rna_subset'].uns['protein_names']

mdata.write(
    DATA_DIR / "03_RNA_and_protein_TotalVI"
    / "objects/PB2/TotalVI_PB2_TEAseq_CLEAN_TF.h5mu"
)